In [13]:
DATASET_PARAMETER_FILES = {
    "mocap": "datasets/mocap/calculated_parameters_v3.json",
    "openpose": "datasets/openpose/calculated_parameters_openpose_triangulated_v2.json",
    "mediapipe": "datasets/mediapipe/calculated_parameters_v4_triangulated.json",
    "hrnet": "datasets/mmpose/calculated_parameters_hrnet_triangulated.json",
    "rtmpose": "datasets/mmpose/calculated_parameters_rtmpose_triangulated.json",
    "vitpose": "datasets/mmpose/calculated_parameters_vitpose_triangulated.json",
    "movenet_lightning": "datasets/movenet/calculated_parameters_lightning_triangulated.json",
    "movenet_thunder": "datasets/movenet/calculated_parameters_thunder_triangulated.json",
    "yolo_v26": "datasets/yolo/calculated_parameters_v3_yolo26.json",
    "yolo_v11": "datasets/yolo/calculated_parameters_v2_triangulated_v3.json",
}

DATASET_PARAMETER_FILES_BUTTERWORTH = datasets_parameters = {
    "mocap": "datasets/mocap/calculated_parameters_butterworth_v3.json",
    "openpose": "datasets/openpose/calculated_parameters_butterworth_openpose_triangulated_v2.json",
    "mediapipe": "datasets/mediapipe/calculated_parameters_butterworth_v4_triangulated.json",
    "hrnet": "datasets/mmpose/calculated_parameters_butterworth_hrnet_triangulated.json",
    "rtmpose": "datasets/mmpose/calculated_parameters_butterworth_rtmpose_triangulated.json",
    "vitpose": "datasets/mmpose/calculated_parameters_butterworth_vitpose_triangulated.json",
    "movenet_lightning": "datasets/movenet/calculated_parameters_butterworth_lightning_triangulated.json",
    "movenet_thunder": "datasets/movenet/calculated_parameters_butterworth_thunder_triangulated.json",
    "yolo_v26": "datasets/yolo/calculated_parameters_butterworth_v3_yolo26.json",
    "yolo_v11": "datasets/yolo/calculated_parameters_butterworth_v2_triangulated_v3.json",
}


PARAMETER_NAMES = [
    "legs_angles",
    "left_knee_angles",
    "right_knee_angles",
    "left_hip_angles",
    "right_hip_angles",
    "left_humerus_angles",
    "right_humerus_angles",
    "left_elbow_angles",
    "right_elbow_angles",
    "ankle_distances",
    "knee_distances",
    "elbow_distances",
    "hand_distances",
    "center_of_gravity_height_change",
    "lateral_pelvic_tilt",
    "pelvis_rotation",
]

In [41]:
import re
import json
from typing import Sequence
import pandas as pd
from sklearn.preprocessing import StandardScaler

def get_person_from_seq_key(sequence_key: str) -> int:
    match = re.search(r"p(\d+)s", sequence_key)
    if match:
        person_id = int(match.group(1))
        return person_id
    else:
        raise Exception(
            f"Person identifier cannot be extracted from provided key ({sequence_key})"
        )

def get_scaler(selected_datasets: Sequence[str],
               training_set: Sequence[int],
               use_butterworth: bool = False):

    scaler = StandardScaler()

    param_ds = {}

    if use_butterworth:
         for name, file_path in DATASET_PARAMETER_FILES_BUTTERWORTH.items():
            with open(file_path, "r", encoding="utf-8") as file:
                param_ds[name] = json.load(file)
    else:
        for name, file_path in DATASET_PARAMETER_FILES.items():
            with open(file_path, "r", encoding="utf-8") as file:
                param_ds[name] = json.load(file)
    
    combined_test_features = {parameter_name:[] for parameter_name in PARAMETER_NAMES}
    
    for dataset_type, sequence_parameters in param_ds.items():
        if dataset_type in selected_datasets:
            for sequence_key, parameters in sequence_parameters.items():
                person_id = get_person_from_seq_key(sequence_key)
                if person_id in training_set:
                    for parameter_name in PARAMETER_NAMES:
                        combined_test_features[parameter_name] += parameters[parameter_name]


    features_df = pd.DataFrame(combined_test_features)
    
    scaler.fit(features_df.values)
    
    return scaler

In [42]:
selected_datasets = ['mocap', 'yolo_v26', 'mediapipe']
training_set = list(range(1, 25))

scaler = get_scaler(selected_datasets = selected_datasets, training_set = training_set)

In [45]:
import numpy as np

with open(DATASET_PARAMETER_FILES['yolo_v26'], "r", encoding="utf-8") as file:
    sequence_parameters = json.load(file)
    
c_idx = 40

stride_matrix = np.array(
    [
        sequence_parameters['p32s1'][parameter][c_idx - 16 : c_idx + 16]
        for parameter in PARAMETER_NAMES
    ]
)

print(stride_matrix[:,0])

[ 3.69095901e+01  1.76855339e+02  1.78752001e+02  1.61696995e+02
  1.64010853e+02  7.12427393e+00  9.91400087e+00  1.64273641e+02
  1.63692301e+02  5.08758672e+02  3.27118845e+02  4.78730017e+02
  5.52048103e+02 -3.29449196e-01 -9.19505433e+01  9.44738976e+01]


In [46]:
scaled_matrix = scaler.transform(stride_matrix.T)
print(scaled_matrix[0])

[ 0.9565512   0.81647761  0.93089727 -0.67753454 -0.49147795 -0.24912791
  0.22861063  0.52148711  0.57674132  0.5220535   0.57109536  0.2222385
  0.27027677 -0.03615872 -1.01037707  0.38967806]
